# Solution ③ - Foundry Diagnostic Settings 獨立驗證 Notebook

這個 notebook **只用來驗證方案 ③**（Foundry resource → 專屬 LAW）。
與 `test-logging.ipynb`（方案 ①）完全獨立，不互相干擾。

## 前置作業
1. 已執行 `scripts\deploy-foundry-diag.ps1` 部署方案 ③ 基礎設施
2. 取得專屬 LAW 名稱（output `foundryLawName`）

## 測試流程
- **TC-07a**：Non-streaming chat → 必有完整 token usage
- **TC-07b**：Streaming chat（不帶 include_usage）→ 預期 token 為 null
- **TC-07c**：Streaming chat（帶 `stream_options.include_usage=true`）→ 驗證 token 是否回傳
- **TC-07d**：用 KQL 對照三筆請求在方案 ③ LAW 中的記錄

In [ ]:
# === 設定 ===
import os, time, uuid, json, getpass
from openai import OpenAI

APIM_ENDPOINT     = 'https://testaigw01.azure-api.net'
APIM_API_PATH     = '/kunlenewfoundry01'   # 同方案 ① 用的 API path
DEPLOYMENT_NAME   = 'Kimi-K2.5'
API_VERSION       = '2024-10-21'
FOUNDRY_LAW_NAME  = '<填入 deploy-foundry-diag.ps1 輸出的 foundryLawName>'

if 'APIM_KEY' not in os.environ:
    os.environ['APIM_KEY'] = getpass.getpass('APIM Subscription Key: ')

client = OpenAI(
    base_url=f'{APIM_ENDPOINT}{APIM_API_PATH}',
    api_key=os.environ['APIM_KEY'],
    default_headers={'api-key': os.environ['APIM_KEY']},
)

# 在每個請求加上唯一 marker，方便 KQL 撈
RUN_ID = f'tc07-{int(time.time())}'
print(f'RUN_ID = {RUN_ID}')

## TC-07a — Non-streaming（基線：必有 token usage）

In [ ]:
marker_a = f'{RUN_ID}-a-nonstream'
resp = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_a}] 用一句話說明什麼是雲原生'}],
    max_tokens=2000,
    stream=False,
)
print('Marker:', marker_a)
print('Usage :', resp.usage)
print('Content head:', (resp.choices[0].message.content or '')[:120])

## TC-07b — Streaming **不帶** include_usage（預期 token 缺失）

In [ ]:
marker_b = f'{RUN_ID}-b-stream-no-usage'
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_b}] 列三個 Azure 服務'}],
    max_tokens=2000,
    stream=True,
)
last_usage = 'NULL (預期)'
for chunk in stream:
    if chunk.usage is not None:
        last_usage = chunk.usage
print('Marker:', marker_b)
print('Last chunk usage:', last_usage)

## TC-07c — Streaming **帶** `stream_options.include_usage=True`
驗證 Kimi-K2.5 是否支援 OpenAI 的 `include_usage` 約定。
若支援 → 最後一個 chunk 會帶完整 usage → 方案 ③ 也會記到。

In [ ]:
marker_c = f'{RUN_ID}-c-stream-with-usage'
stream = client.chat.completions.create(
    model=DEPLOYMENT_NAME,
    messages=[{'role':'user','content': f'[{marker_c}] 用一句話介紹 APIM'}],
    max_tokens=2000,
    stream=True,
    stream_options={'include_usage': True},
)
last_usage = None
chunks_seen = 0
for chunk in stream:
    chunks_seen += 1
    if chunk.usage is not None:
        last_usage = chunk.usage
print('Marker:', marker_c)
print('Total chunks:', chunks_seen)
print('Final usage :', last_usage)
if last_usage is None:
    print('⚠️  Kimi-K2.5 不支援 include_usage → 方案 ③ 對 streaming 將拿不到 token')
else:
    print('✅  Kimi-K2.5 支援 include_usage → 方案 ③ 可記到 streaming token')

## TC-07d — 等 5-10 分鐘後在【方案 ③ LAW】查詢

**手動步驟**：
1. Azure Portal → Log Analytics workspaces → 選 `log-foundry-diag-*`
2. Logs → 貼下方 query → 把 `<RUN_ID>` 換成 cell 1 印出的值

```kusto
AzureDiagnostics
| where TimeGenerated > ago(30m)
| where Category == "RequestResponse"
| extend P = parse_json(properties_s)
| extend Usage = P.usage
| project TimeGenerated,
          OperationName,
          DurationMs,
          ResultSignature,
          Model            = tostring(P.modelDeploymentName),
          PromptTokens     = toint(Usage.prompt_tokens),
          CompletionTokens = toint(Usage.completion_tokens),
          ReasoningTokens  = toint(Usage.completion_tokens_details.reasoning_tokens),
          TotalTokens      = toint(Usage.total_tokens)
| order by TimeGenerated desc
| take 20
```

**預期結果**：
| 測試 | TotalTokens 欄位 |
|---|---|
| TC-07a (non-stream) | ✅ 數字 |
| TC-07b (stream, no include_usage) | ⚠️ null |
| TC-07c (stream, include_usage) | ✅/⚠️ 視 Kimi 是否支援 |

## 並排比較方案 ① vs ③
切到方案 ① LAW（`log-aigw-*`），對同一段時間跑：
```kusto
AppDependencies
| where TimeGenerated > ago(30m)
| where Name has "chat/completions"
| extend P = parse_json(tostring(Properties))
| project TimeGenerated, ResultCode, DurationMs,
          ReqSize  = strlen(tostring(P["Request-Body"])),
          RespSize = strlen(tostring(P["Response-Body"])),
          Truncated = strlen(tostring(P["Response-Body"])) >= 262000
| order by TimeGenerated desc
```

對比兩邊筆數、時間、status code 是否一致 → 確認雙軌平行運作。